In [11]:
import plotly.express as px
import pandas as pd
import numpy as np
from scipy.stats import gaussian_kde
import plotly.graph_objects as go

1. Is there a correlation between graduation rates and number of economically disadvantaged students in each district?

In [7]:
df = pd.read_csv(r"..\data\processed\GRAD_PCT_vs_PER_ECDIS.csv")
df

,AGGREGATION_CODE,AGGREGATION_NAME,ENROLL_CNT,GRAD_CNT,GRAD_PCT,PER_ECDIS
0,240101040000,AVON CSD,73,69,0.95,0.34
1,240901040000,MT MORRIS CSD,54,50,0.93,0.64
2,241001060000,DANSVILLE CSD,122,100,0.82,0.55
3,430901060000,GORHAM-MIDDLESEX CSD (MARCUS WHITMAN,93,81,0.87,0.46
4,241101040000,DALTON-NUNDA CSD (KESHEQUA),52,47,0.90,0.47
...,...,...,...,...,...,...
663,580912060000,EASTPORT-SOUTH MANOR CSD,285,257,0.90,0.28
664,10623060000,NORTH COLONIE CSD,503,468,0.93,0.31
665,212101040000,CENTRAL VALLEY CSD AT ILION-MOHAWK,166,133,0.80,0.49
666,271201040000,OPPENHEIM-EPHRATAH-ST. JOHNSVILLE CS,53,47,0.89,0.64


In [9]:
# Remove outlier rows based on enrollment count
lower = df["ENROLL_CNT"].quantile(0.10)
upper = df["ENROLL_CNT"].quantile(0.90)

df = df[(df["ENROLL_CNT"] >= lower) & (df["ENROLL_CNT"] <= upper)]
df

,AGGREGATION_CODE,AGGREGATION_NAME,ENROLL_CNT,GRAD_CNT,GRAD_PCT,PER_ECDIS
0,240101040000,AVON CSD,73,69,0.95,0.34
1,240901040000,MT MORRIS CSD,54,50,0.93,0.64
2,241001060000,DANSVILLE CSD,122,100,0.82,0.55
3,430901060000,GORHAM-MIDDLESEX CSD (MARCUS WHITMAN,93,81,0.87,0.46
4,241101040000,DALTON-NUNDA CSD (KESHEQUA),52,47,0.90,0.47
...,...,...,...,...,...,...
662,571502060000,CANISTEO-GREENWOOD CSD,89,85,0.96,0.58
663,580912060000,EASTPORT-SOUTH MANOR CSD,285,257,0.90,0.28
664,10623060000,NORTH COLONIE CSD,503,468,0.93,0.31
665,212101040000,CENTRAL VALLEY CSD AT ILION-MOHAWK,166,133,0.80,0.49


In [10]:
fig = px.scatter(
    df,
    x="GRAD_PCT",
    y="PER_ECDIS",
    color="ENROLL_CNT",
    color_continuous_scale="Viridis",
    hover_name="AGGREGATION_NAME",
    labels={
        "GRAD_PCT": "Graduation Percentage",
        "PER_ECDIS": "Percent Economically Disadvantaged",
        "ENROLL_CNT": "Enrollment Count",
        "AGGREGATION_NAME": "District"
    },
    title="Graduation Rate vs Percentage Economically Disadvantaged"
)

fig.update_layout(
    template="plotly_white"
)

fig.show()

In [11]:
fig.write_html("../results/grad_rate_VS_per_ecdis.html")

2. What are the grade distributions for charter vs non-charter schools in each subject?

In [9]:
df = pd.read_csv(r"..\data\processed\charter_vs_noncharter_PER_PROF.csv")
df

,INSTITUTION_ID,SCHOOL_NAME,SCHOOL_TYPE,SUBJECT,TESTED,PER_PROF
0,800000055741,WILLIAM S HACKETT MIDDLE SCHOOL,District,Regents Life Science: Biology,67,82
1,800000055743,ALBANY HIGH SCHOOL,District,Regents Common Core Algebra II,176,90
2,800000055743,ALBANY HIGH SCHOOL,District,Regents US History&Gov't,870,54
3,800000055743,ALBANY HIGH SCHOOL,District,Regents Phy Set/Physics,90,42
4,800000055743,ALBANY HIGH SCHOOL,District,Regents Phy Set/Chemistry,234,47
...,...,...,...,...,...,...
9248,800000034430,DUNDEE JUNIOR-SENIOR HIGH SCHOOL,District,Regents NF Global History,60,63
9249,800000034430,DUNDEE JUNIOR-SENIOR HIGH SCHOOL,District,Regents Phy Set/Chemistry,13,100
9250,800000034430,DUNDEE JUNIOR-SENIOR HIGH SCHOOL,District,Regents Common Core Algebra II,15,80
9251,800000034430,DUNDEE JUNIOR-SENIOR HIGH SCHOOL,District,Regents Common Core English Language Art,39,77


In [10]:
df.SCHOOL_TYPE.unique()

<StringArray>
['District', 'Charter']
Length: 2, dtype: str

In [19]:
# ── Configuration ───────────────────────────────────────────────────
subjects = df["SUBJECT"].unique().tolist()  # 8 subjects
school_types = {
    "District": "rgba(70, 130, 180, 0.7)",   # blue
    "Charter": "rgba(230, 100, 100, 0.7)",    # red/salmon
}
line_colors = {
    "District": "rgba(70, 130, 180, 1)",
    "Charter": "rgba(230, 100, 100, 1)",
}

# Spacing between ridgeline rows
row_height = 0.6

# ── KDE helper (weighted by TESTED) ────────────────────────────────
def weighted_kde(values, weights, x_grid, bw_method=0.3):
    """Compute a weighted KDE over x_grid."""
    # Repeat values proportionally to weights to simulate weighting
    weights = np.asarray(weights, dtype=float)
    weights = weights / weights.sum()  # normalize

    try:
        kde = gaussian_kde(values, bw_method=bw_method, weights=weights)
        density = kde(x_grid)
        return density
    except Exception:
        return np.zeros_like(x_grid)


# ── Build the figure ────────────────────────────────────────────────
fig = go.Figure()

x_grid = np.linspace(0, 100, 500)  # PER_PROF range 0–100%

for i, subject in enumerate(subjects):
    baseline = i * row_height  # vertical offset for this row

    for school_type, fill_color in school_types.items():
        subset = df[
            (df["SUBJECT"] == subject) & (df["SCHOOL_TYPE"] == school_type)
        ]

        if len(subset) < 2:
            continue

        values = subset["PER_PROF"].values
        weights = subset["TESTED"].values

        density = weighted_kde(values, weights, x_grid, bw_method=0.2)

        # Scale density so peaks look nice (adjust multiplier as needed)
        density_scaled = density / density.max() * (row_height * 0.9)

        # Shifted y-values sitting on each row's baseline
        y_vals = baseline + density_scaled

        # Fill area: trace the curve, then return along the baseline
        fig.add_trace(go.Scatter(
            x=np.concatenate([x_grid, x_grid[::-1]]),
            y=np.concatenate([y_vals, np.full_like(x_grid, baseline)]),
            fill="toself",
            fillcolor=fill_color,
            line=dict(color=line_colors[school_type], width=1),
            name=school_type,
            legendgroup=school_type,
            showlegend=(i == 0),  # show legend only once per type
            hoverinfo="skip",
        ))

    # Baseline separator line for each subject row
    fig.add_shape(
        type="line",
        x0=0, x1=100,
        y0=baseline, y1=baseline,
        line=dict(color="black", width=0.5),
    )

# ── Layout ──────────────────────────────────────────────────────────
fig.update_layout(
    title="Proficiency Distribution by Subject",
    xaxis=dict(
        title="Percentage Proficiency (PER_PROF)",
        range=[0, 100],
        showgrid=False,
    ),
    yaxis=dict(
        tickvals=[i * row_height for i in range(len(subjects))],
        ticktext=subjects,
        showgrid=False,
        zeroline=False,
    ),
    plot_bgcolor="white",
    legend=dict(
        title="School Type",
        x=0.85, y=1.0,
    ),
    height=900,
    width=700,
    margin=dict(l=120, r=40, t=60, b=60),
)

fig.show()